# 🚀 ADAN Trading Bot - Google Colab T4 GPU Pipeline

**Complete training pipeline optimized for Google Colab T4 GPU**

## Pipeline Overview

1. **GPU Verification** - Confirm T4 GPU allocation
2. **Setup** - Clone repo & install dependencies
3. **Data Download** - 50,000 real BTC/USDT candles via CCXT
4. **Training** - PBT with 4 worker profiles on T4
5. **Model Extraction** - Best model to production
6. **Paper Trading** - Isolated virtual wallet ($20.50)
7. **Validation** - Trade lifecycle checks

---

**Architecture:**
- PPO + ContextualTemporalFusionExtractor (FiLM Meta-RL)
- Multi-timeframe: 5m (master), 1h, 4h
- Capital Tiers: Micro ($11-$30) → Enterprise ($1000+)
- Virtual wallet: $20.50 initial, 100% isolated

**T4 GPU Config:**
- 4 workers share 1 T4 (16GB VRAM)
- resources_per_trial: {cpu: 0.5, gpu: 0.25}
- Auto-detected by train_parallel_agents.py


## 0. 🔍 Verify GPU Allocation

**IMPORTANT:** Make sure you have a T4 GPU allocated.

Go to: **Runtime → Change runtime type → Hardware accelerator → GPU (T4)**


In [ ]:
# Verify GPU availability and specs
import torch
import subprocess

print("=" * 70)
print("  GPU VERIFICATION")
print("=" * 70)

print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"\n✅ GPU detected:")
    print(f"   Name: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"   VRAM: {props.total_memory / 1e9:.1f} GB")
    print(f"   Compute capability: {props.major}.{props.minor}")
    
    # Check if it's a T4
    gpu_name = torch.cuda.get_device_name(0)
    if "T4" in gpu_name:
        print(f"\n✅ T4 GPU confirmed - ready for training!")
    else:
        print(f"\n⚠️  Warning: Expected T4, got {gpu_name}")
        print(f"   Training will work but may be slower.")
else:
    print("\n❌ NO GPU DETECTED!")
    print("\n   Go to: Runtime → Change runtime type → GPU (T4)")
    print("   Then restart this notebook.")
    raise RuntimeError("GPU required for training")

# Show nvidia-smi
print("\n" + "=" * 70)
print("  NVIDIA-SMI OUTPUT")
print("=" * 70)
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

## 1. 📦 Setup - Clone Repository & Install Dependencies


In [ ]:
# Clone ADAN repository
import os

REPO_URL = "https://github.com/Cabrel10/ADAN0.git"
REPO_DIR = "/content/ADAN0"

print("=" * 70)
print("  CLONING REPOSITORY")
print("=" * 70)

if os.path.exists(REPO_DIR):
    print(f"\nRepository exists at {REPO_DIR}")
    %cd {REPO_DIR}
    print("\nPulling latest changes...")
    !git pull origin main
else:
    print(f"\nCloning from {REPO_URL}...")
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

print(f"\n✅ Working directory: {os.getcwd()}")
print("\nLatest commits:")
!git log --oneline -5

In [ ]:
# Install dependencies
print("=" * 70)
print("  INSTALLING DEPENDENCIES")
print("=" * 70)

# Install from requirements.txt
print("\nInstalling from requirements.txt...")
!pip install -q -r requirements.txt

# Install additional packages for Colab
print("\nInstalling Colab-specific packages...")
!pip install -q ccxt pandas_ta 'ray[tune]>=2.5.0' pyarrow fastparquet

# Install ADAN in editable mode
print("\nInstalling ADAN package in editable mode...")
!pip install -q -e .

print("\n" + "=" * 70)
print("  VERIFYING IMPORTS")
print("=" * 70)

# Verify key imports
try:
    import sys
    sys.path.insert(0, 'src')
    
    from adan_trading_bot.common.config_loader import ConfigLoader
    from adan_trading_bot.environment.multi_asset_chunked_env import MultiAssetChunkedEnv
    from adan_trading_bot.portfolio.portfolio_manager import PortfolioManager
    import ccxt
    import ray
    import stable_baselines3
    
    print("\n✅ All imports successful!")
    print(f"   - CCXT: {ccxt.__version__}")
    print(f"   - Ray: {ray.__version__}")
    print(f"   - SB3: {stable_baselines3.__version__}")
    print(f"   - PyTorch: {torch.__version__}")
    
except ImportError as e:
    print(f"\n❌ Import error: {e}")
    raise

## 2. 📊 Data Download - 50,000 Real Candles via CCXT

Downloads BTC/USDT candles from public exchanges (no API key needed).

The script tries exchanges in order:
1. Binance (primary)
2. Bybit (fallback)
3. Bitget (fallback)
4. Kraken (fallback)


In [ ]:
# Download 50,000 candles for BTCUSDT (all timeframes: 5m, 1h, 4h)
print("=" * 70)
print("  DOWNLOADING MARKET DATA VIA CCXT")
print("=" * 70)

print("\nDownloading 50,000 candles for BTCUSDT...")
print("Timeframes: 5m (master clock), 1h, 4h")
print("This may take 5-10 minutes...\n")

!python scripts/download_ccxt_data.py \
    --symbols BTCUSDT \
    --timeframes 5m 1h 4h \
    --limit 50000 \
    --output data/raw/ccxt

print("\n" + "=" * 70)
print("  PROCESSING INDICATORS")
print("=" * 70)

# Process indicators
!python scripts/process_indicators.py \
    --input data/raw/ccxt \
    --output data/processed/indicators/train

print("\n" + "=" * 70)
print("  VERIFYING DATA")
print("=" * 70)

# Verify the downloaded data
import pandas as pd
import glob

data_dir = "data/processed/indicators/train/BTCUSDT"
if os.path.exists(data_dir):
    for tf in ["5m", "1h", "4h"]:
        file_path = f"{data_dir}/{tf}.parquet"
        if os.path.exists(file_path):
            df = pd.read_parquet(file_path)
            print(f"\n✅ {tf}: {len(df):,} rows, {len(df.columns)} columns")
            print(f"   Date range: {df.index.min()} → {df.index.max()}")
            print(f"   Columns: {', '.join(df.columns[:10])}...")
        else:
            print(f"\n❌ {tf}: File not found at {file_path}")
else:
    print(f"\n❌ Data directory not found: {data_dir}")
    print("   Check download_ccxt_data.py output for errors.")

## 3. 🎯 Training - PBT with 4 Worker Profiles on T4 GPU

**Auto-detected Colab T4 configuration:**
- `num_cpus=2` (Colab limit)
- `device="cuda"` (T4 GPU)
- `resources_per_trial={cpu: 0.5, gpu: 0.25}` (4 workers share T4)

**Worker profiles:**
- **Scalper** (5m): gamma=0.95, n_steps=512, fast trades
- **Intraday** (1h): gamma=0.99, n_steps=2048, day trades
- **Swing** (4h): gamma=0.995, n_steps=8192, multi-day
- **Position** (4h): gamma=0.999, n_steps=16384, long-term

**Training options:**
- **Quick test:** 100K steps (~10 min)
- **Full training:** 1M steps (~2 hours)


In [ ]:
# Option A: Quick training test (100K steps, ~10 minutes)
print("=" * 70)
print("  QUICK TRAINING TEST (100K steps)")
print("=" * 70)

!python scripts/train_parallel_agents.py \
    --config config/config.yaml \
    --steps 100000 \
    --profiles scalper intraday swing position \
    --num-samples 4 \
    --num-cpus 2 \
    --steps-per-iter 5000 \
    --no-subproc

In [ ]:
# Option B: Full training (1M steps, ~2 hours)
# Uncomment to run full training

# print("=" * 70)
# print("  FULL TRAINING (1M steps)")
# print("=" * 70)

# !python scripts/train_parallel_agents.py \
#     --config config/config.yaml \
#     --steps 1000000 \
#     --profiles scalper intraday swing position \
#     --num-samples 4 \
#     --num-cpus 2 \
#     --steps-per-iter 10000 \
#     --no-subproc

## 4. 🏆 Model Extraction - Best Model to Production

Scans all Ray Tune results and extracts:
- Trial with highest mean_reward
- Model checkpoint (model.zip)
- VecNormalize stats (vecnormalize.pkl)
- Metadata (extraction_metadata.json)

Output: `models/rl_agents/production/`


In [ ]:
# Extract the best model from training results
print("=" * 70)
print("  EXTRACTING BEST MODEL")
print("=" * 70)

!python scripts/extract_best_model.py --metric mean_reward

print("\n" + "=" * 70)
print("  VERIFYING EXTRACTION")
print("=" * 70)

# Verify extraction
import json

prod_dir = "models/rl_agents/production"
model_path = f"{prod_dir}/model.zip"
meta_path = f"{prod_dir}/extraction_metadata.json"

if os.path.exists(model_path):
    size_mb = os.path.getsize(model_path) / 1e6
    print(f"\n✅ Production model extracted:")
    print(f"   Path: {model_path}")
    print(f"   Size: {size_mb:.1f} MB")
    
    if os.path.exists(meta_path):
        with open(meta_path) as f:
            meta = json.load(f)
        print(f"\n   Metadata:")
        print(f"   - Source trial: {meta.get('source_trial', '?')}")
        print(f"   - Mean reward: {meta.get('mean_reward', 0):.4f}")
        print(f"   - Mean Sharpe: {meta.get('mean_sharpe', 0):.4f}")
        print(f"   - Total steps: {meta.get('total_steps', 0):,}")
        print(f"   - Extraction time: {meta.get('extraction_time', '?')}")
else:
    print(f"\n❌ No production model found at {model_path}")
    print("   Check training output for errors.")
    print("   Training may have failed or not completed.")

## 5. 📈 Paper Trading - Isolated Virtual Wallet

**CRITICAL: 100% Isolated Virtual Wallet**
- Initial balance: **$20.50**
- Max balance: **$25.00** (Micro Capital tier)
- Max concurrent positions: **1**
- **NO real orders are EVER placed**
- **NO interaction with real exchanges**
- Uses local data for simulation

**Logs:**
- `[INTENTION]` - What the agent wants to do
- `[EXECUTION]` - What actually happens (virtual)
- 1000% auditable, zero risk


In [ ]:
# Run paper trading with isolated virtual wallet
print("=" * 70)
print("  PAPER TRADING - ISOLATED VIRTUAL WALLET")
print("=" * 70)

print("\n⚠️  IMPORTANT: This is 100% virtual simulation")
print("   - Initial balance: $20.50")
print("   - Max balance: $25.00")
print("   - NO real orders placed")
print("   - Uses local data only\n")

# Run for 10 minutes using local data
!python scripts/paper_trading_monitor.py \
    --model models/rl_agents/production/model.zip \
    --duration 10 \
    --initial-balance 20.50 \
    --max-balance 25.00 \
    --max-positions 1

print("\n" + "=" * 70)
print("  PAPER TRADING RESULTS")
print("=" * 70)

# Show results if available
report_path = "results/paper_trading_report.json"
if os.path.exists(report_path):
    with open(report_path) as f:
        report = json.load(f)
    
    print(f"\n✅ Paper Trading Summary:")
    print(f"   - Total trades: {report.get('total_trades', 0)}")
    print(f"   - Win rate: {report.get('win_rate_pct', 0):.1f}%")
    print(f"   - Total return: {report.get('total_return_pct', 0):+.2f}%")
    print(f"   - Sharpe ratio: {report.get('sharpe_ratio', 0):.2f}")
    print(f"   - Max drawdown: {report.get('max_drawdown_pct', 0):.2f}%")
    print(f"   - Final balance: ${report.get('final_balance', 0):.2f}")
else:
    print(f"\n⚠️  No report found at {report_path}")
    print("   Check paper_trading_monitor.py output for errors.")

## 6. ✅ Validation - Trade Lifecycle Checks

Validates that all trading rules are enforced:
- A. RESET - Episode initialization
- B. TARGET_WEIGHT - Action decoded
- C. TRADE_OPEN - Position opened
- D. HOLD_MIN - Minimum hold enforced
- E. POSITION_CLOSE - SL/TP/Agent close
- F. OPEN_CLOSE_RATIO - 1:1 ratio
- G. WAIT_BLOCK - Post-SL/TP cooldown
- H. REWARD_ANTIHACK - Reward formula
- I-L. Additional checks


In [ ]:
# Run a quick training and validate lifecycle
print("=" * 70)
print("  TRADE LIFECYCLE VALIDATION")
print("=" * 70)

print("\nRunning quick training (1000 steps) for validation...\n")

!python scripts/train_simple_ppo.py --steps 1000 2>&1 | tee /tmp/validation_train.log

print("\n" + "=" * 70)
print("  RUNNING LIFECYCLE VALIDATOR")
print("=" * 70 + "\n")

!python scripts/validate_trade_lifecycle.py /tmp/validation_train.log --run-id "Colab-Validation"

## 7. 📦 Results & Download

Package and download the trained model for local use.


In [ ]:
# Package production model for download
import shutil
from datetime import datetime

print("=" * 70)
print("  PACKAGING RESULTS FOR DOWNLOAD")
print("=" * 70)

prod_dir = "models/rl_agents/production"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
archive_name = f"adan_production_model_{timestamp}"
archive_path = f"/content/{archive_name}"

if os.path.exists(prod_dir) and os.path.exists(f"{prod_dir}/model.zip"):
    # Create archive
    shutil.make_archive(archive_path, "zip", prod_dir)
    
    archive_size = os.path.getsize(f"{archive_path}.zip") / 1e6
    print(f"\n✅ Model packaged successfully:")
    print(f"   Path: {archive_path}.zip")
    print(f"   Size: {archive_size:.1f} MB")
    print(f"\n   Download from Files panel (📁 icon) in Colab.")
    print(f"   Or use: files.download('{archive_path}.zip')")
    
    # Show what's included
    print(f"\n   Contents:")
    for item in os.listdir(prod_dir):
        item_path = os.path.join(prod_dir, item)
        if os.path.isfile(item_path):
            size = os.path.getsize(item_path) / 1e6
            print(f"   - {item} ({size:.1f} MB)")
else:
    print(f"\n❌ No production model to package.")
    print(f"   Directory: {prod_dir}")
    print(f"   Make sure training completed successfully.")

# Summary
print("\n" + "=" * 70)
print("  PIPELINE COMPLETE")
print("=" * 70)
print("\n✅ All steps completed successfully!")
print("\nNext steps:")
print("1. Download the model archive from Files panel")
print("2. Extract locally and load with:")
print("   from stable_baselines3 import PPO")
print("   model = PPO.load('path/to/model.zip')")
print("3. Use for paper trading or further training")
print("\n" + "=" * 70)